# Deep Learning Image Preprocessing



In [12]:
import sys
sys.path.append('/host/d/Github/')

import os
import numpy as np
import pandas as pd
import nibabel as nb

import Osteosarcoma.Data_processing as Data_processing
import Osteosarcoma.Build_lists.Build_list as Build_list
import Osteosarcoma.functions_collection as ff

### Step 0: imports and settings


In [13]:

# ============================================================
# Paths
# ============================================================

patient_list_file = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx'
original_data_root = '/host/e/D/Data/Habitats/Jishuitan/original_data'
slice_data_root = '/host/e/D/Data/Habitats/Jishuitan/resampled_data_new'
patient_list_out_dir = '/host/e/D/Data/Habitats/Jishuitan/Patient_lists'

os.makedirs(slice_data_root, exist_ok=True)
os.makedirs(patient_list_out_dir, exist_ok=True)



build = Build_list.Build(patient_list_file)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()

print('Number of cases to process:', len(patient_index_list))
print('Example image:', image_path_list[0])
print('Example mask :', mask_path_list[0])


Number of cases to process: 330
Example image: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz
Example mask : /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz


### step 1: for each patient, find out the slice with the largest tumor area, and save it as a new image file. The new image file will be saved in the `slice_data_root` directory, with the same name as the original image file.

#### we should also save the patient list with which slice is selected, and the corresponding slice image shape

In [19]:
result_list = []
for i in range(len(patient_index_list)):
    patient_index = patient_index_list[i]
    patient_set = patient_set_list[i]
    print('Processing patient:', patient_index, 'in set:', patient_set)
    image_path = os.path.join(slice_data_root, patient_set, patient_index, 'img.nii.gz')
    mask_path = os.path.join(slice_data_root, patient_set, patient_index, 'label.nii.gz')

    new_image_path = os.path.join(slice_data_root, patient_set, patient_index, 'img_slices.nii.gz')
    new_mask_path = os.path.join(slice_data_root, patient_set, patient_index, 'label_slices.nii.gz')
    new_bbox_path = os.path.join(slice_data_root, patient_set, patient_index, 'bbox_mask_slices.nii.gz')

    if os.path.exists(new_bbox_path):
        print(f'Patient {patient_index} already processed, skipping.')
        # load the existing slice to get its shape
    else:

        # load the image and mask
        nii = nb.load(image_path)
        ii = nii.get_fdata()
        affine = nii.affine

        mask_nii = nb.load(mask_path)
        mask_ii = mask_nii.get_fdata()

        # find the slice with the largest tumor area
        tumor_area_list = []
        for j in range(ii.shape[2]):
            tumor_area = np.sum(mask_ii[:,:,j])
            tumor_area_list.append(tumor_area)
        
        max_tumor_slice_index = np.argmax(tumor_area_list)

        max_tumore_3slices = [max_tumor_slice_index-1, max_tumor_slice_index, max_tumor_slice_index+1]
        new_image = ii[:,:,max_tumore_3slices]
        new_mask = mask_ii[:,:,max_tumore_3slices]

        # save the slice with the largest tumor area as a new image file, also save the corresponding mask slice
        new_image_nii = nb.Nifti1Image(new_image, affine)
        new_mask_nii = nb.Nifti1Image(new_mask, affine)
        ff.make_folder([os.path.join(slice_data_root, patient_set), os.path.join(slice_data_root, patient_set, patient_index)])
        nb.save(new_image_nii, new_image_path)
        nb.save(new_mask_nii, new_mask_path)

        bbox_arr, x0, x1, y0, y1, z0, z1 = Data_processing.bbox3d(
            new_mask,
            buffer_x=5,
            buffer_y=5,
            buffer_z=0)
        nb.save(nb.Nifti1Image(bbox_arr, affine), new_bbox_path)

    # load
    new_image_nii = nb.load(new_image_path).get_fdata()
    new_bbox_nii = nb.load(new_bbox_path).get_fdata()
    slice_shape_x, slice_shape_y, slice_shape_z = new_image_nii.shape
    
    bbox_region = np.where(new_bbox_nii > 0)
    tumor_slice_shape_x = bbox_region[0].max() - bbox_region[0].min() + 1
    tumor_slice_shape_y = bbox_region[1].max() - bbox_region[1].min() + 1
    result_list.append([patient_index, patient_set, max_tumor_slice_index, slice_shape_x, slice_shape_y, tumor_slice_shape_x, tumor_slice_shape_y])

    df = pd.DataFrame(result_list, columns=['patient_index', 'patient_set', 'max_tumor_slice_index', 'slice_shape_x', 'slice_shape_y', 'tumor_slice_shape_x', 'tumor_slice_shape_y'])
    df.to_excel(os.path.join(patient_list_out_dir, 'largest_slice_info_set12.xlsx'), index=False)

Processing patient: 1 in set: set_1
Processing patient: 5 in set: set_1
Processing patient: 7 in set: set_1
Processing patient: 8 in set: set_1
Processing patient: 11 in set: set_1
Processing patient: 15 in set: set_1
Processing patient: 18 in set: set_1
Processing patient: 19 in set: set_1
Processing patient: 20 in set: set_1
Processing patient: 21 in set: set_1
Processing patient: 22 in set: set_1
Processing patient: 23 in set: set_1
Processing patient: 24 in set: set_1
Processing patient: 26 in set: set_1
Processing patient: 28 in set: set_1
Processing patient: 29 in set: set_1
Processing patient: 30 in set: set_1
Processing patient: 33 in set: set_1
Processing patient: 34 in set: set_1
Processing patient: 36 in set: set_1
Processing patient: 38 in set: set_1
Processing patient: 40 in set: set_1
Processing patient: 41 in set: set_1
Processing patient: 42 in set: set_1
Processing patient: 43 in set: set_1
Processing patient: 44 in set: set_1
Processing patient: 46 in set: set_1
Proce

### Step 2: N4 bias field correction

In [7]:
import os
import SimpleITK as sitk

resampled_data_root = '/host/e/D/Data/Habitats/Jishuitan/largest_slice'

# ============================================================
# N4 settings
# ============================================================

n4_max_iterations = [50, 50, 30, 20]


# ============================================================
# Run all cases
# ============================================================

for i in range(0, len(patient_index_list)):
    patient_set = patient_set_list[i]
    patient_index = patient_index_list[i]

    print("\n============================================================")
    print("Processing patient set:", patient_set, "patient index:", patient_index, " i is ", i)

    case_resampled_dir = os.path.join(
        resampled_data_root,
        str(patient_set),
        str(patient_index),
    )

    img_path = os.path.join(case_resampled_dir, "img.nii.gz")
    img_n4_path = os.path.join(case_resampled_dir, "img_n4.nii.gz")

    if not os.path.isfile(img_path):
        print("  Resampled image not found. Skipping:", img_path)
        continue

    if os.path.isfile(img_n4_path):
        print("  N4 image already exists. Skipping:", img_n4_path)
        continue

    try:
        # ----------------------------------------------------
        # 1. Read image
        # ----------------------------------------------------
        img_sitk = sitk.ReadImage(img_path, sitk.sitkFloat32)

        # ----------------------------------------------------
        # 2. Build a simple foreground mask
        # ----------------------------------------------------
        mask_sitk = sitk.OtsuThreshold(img_sitk, 0, 1, 200)

        # ----------------------------------------------------
        # 3. Run N4 bias correction
        # ----------------------------------------------------
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations(n4_max_iterations)

        print("  Running N4...")
        img_n4_sitk = corrector.Execute(img_sitk, mask_sitk)

        # ----------------------------------------------------
        # 4. Save corrected image
        # ----------------------------------------------------
        sitk.WriteImage(img_n4_sitk, img_n4_path)

        print("  Saved:", img_n4_path)

    except Exception as e:
        print("  Failed:", str(e))


Processing patient set: set_1 patient index: 1  i is  0
  Running N4...
  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_1/1/img_n4.nii.gz

Processing patient set: set_1 patient index: 5  i is  1
  Running N4...


NiftiImageIO (0x58320b3d60e0): Non-orthogonal direction matrix coerced to orthogonal



  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_1/5/img_n4.nii.gz

Processing patient set: set_1 patient index: 7  i is  2
  Running N4...


NiftiImageIO (0x58320b3d60e0): Non-orthogonal direction matrix coerced to orthogonal



  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_1/7/img_n4.nii.gz

Processing patient set: set_1 patient index: 8  i is  3
  Running N4...


...

  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_2/284/img_n4.nii.gz

Processing patient set: set_2 patient index: 288  i is  327
  Running N4...


NiftiImageIO (0x58320b3d60e0): Non-orthogonal direction matrix coerced to orthogonal



  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_2/288/img_n4.nii.gz

Processing patient set: set_2 patient index: 290  i is  328
  Running N4...


NiftiImageIO (0x58320b3d60e0): Non-orthogonal direction matrix coerced to orthogonal



  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_2/290/img_n4.nii.gz

Processing patient set: set_2 patient index: 294  i is  329
  Running N4...
  Saved: /host/e/D/Data/Habitats/Jishuitan/largest_slice/set_2/294/img_n4.nii.gz
